⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# `HR_TRG_EMP_Full_Load.sql` Conversion\n",
        "\n",
        "**Converted On:** 2024-07-30\n",
        "\n",
        "**Description:** This notebook converts an ODI SQL script for a full load insert into the `TRG_EMP` table from the `EMPLOYEES` table in the HR schema. It includes standard Databricks widget setup and schema/syntax conversions."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"ETL Job Type (e.g., L for Load)\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"-1\", \"Datasource Number ID\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"-1\", \"ETL Process Widget ID\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"-1\", \"ODI Session Number\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# ETL Parameters\n",
        "\n",
        "The following temporary views are created to make ETL parameter values accessible in SQL cells."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS\n",
        "SELECT '${ETL_JOB_TYPE}' AS etl_job_type;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS\n",
        "SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS\n",
        "SELECT ${ETL_PROC_WID} AS etl_proc_wid;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS\n",
        "SELECT '${ODI_SESS_NO}' AS odi_sess_no;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "etl_job_type = dbutils.widgets.get(\"ETL_JOB_TYPE\")\n",
        "datasource_num_id = dbutils.widgets.get(\"DATASOURCE_NUM_ID\")\n",
        "etl_proc_wid = dbutils.widgets.get(\"ETL_PROC_WID\")\n",
        "odi_sess_no = dbutils.widgets.get(\"ODI_SESS_NO\")\n",
        "\n",
        "print(f\"ETL_JOB_TYPE: {etl_job_type}\")\n",
        "print(f\"DATASOURCE_NUM_ID: {datasource_num_id}\")\n",
        "print(f\"ETL_PROC_WID: {etl_proc_wid}\")\n",
        "print(f\"ODI_SESS_NO: {odi_sess_no}\")\n",
        "\n",
        "display(spark.sql(\"SELECT * FROM v_etl_job_type\"))\n",
        "display(spark.sql(\"SELECT * FROM v_datasource_num_id\"))\n",
        "display(spark.sql(\"SELECT * FROM v_etl_proc_wid\"))\n",
        "display(spark.sql(\"SELECT * FROM v_odi_sess_no\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Insert into Target\n",
        "\n",
        "This section performs the full load insertion into the target table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO in {30}\n",
        "INSERT INTO workspace.hr.trg_emp\n",
        "  (\n",
        "    employee_id ,\n",
        "    first_name ,\n",
        "    last_name ,\n",
        "    email ,\n",
        "    phone_number ,\n",
        "    hire_date ,\n",
        "    job_id ,\n",
        "    salary ,\n",
        "    commission_pct ,\n",
        "    manager_id ,\n",
        "    department_id\n",
        "  )\n",
        "SELECT\n",
        "  employees.employee_id ,\n",
        "  employees.first_name ,\n",
        "  employees.last_name ,\n",
        "  employees.email ,\n",
        "  employees.phone_number ,\n",
        "  employees.hire_date ,\n",
        "  employees.job_id ,\n",
        "  employees.salary ,\n",
        "  employees.commission_pct ,\\
        "  employees.manager_id ,\n",
        "  employees.department_id\n",
        "FROM\n",
        "  workspace.hr.employees AS employees;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) AS records_inserted FROM workspace.hr.trg_emp;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Validation\n",
        "\n",
        "Final checks to ensure data integrity and expected row counts."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "collapsed": false
      },
      "source": [
        "-- MAGIC %sql\n",
        "SELECT\n",
        "  'Target Table' AS table_name,\n",
        "  COUNT(*) AS total_rows\n",
        "FROM\n",
        "  workspace.hr.trg_emp;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Conversion Notes\n",
        "\n",
        "**General Notes:**\n",
        "- The original ODI SQL contained only a direct `INSERT` statement. No staging (`C$`), flow (`I$`), or error (`E$`) tables were present in the source.\n",
        "- Oracle hints `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Databricks Delta Lake and Spark SQL.\n",
        "- Oracle schema `HR` has been converted to `workspace.hr` as per naming conventions.\n",
        "- Table names `TRG_EMP` and `EMPLOYEES` have been lowercased to `trg_emp` and `employees`.\n",
        "- Databricks widgets for `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, and `ODI_SESS_NO` have been added as boilerplate, although they are not used in the provided `INSERT` statement.\n",
        "\n",
        "**Manual Actions Required:**\n",
        "- **Table DDL:** Ensure that `workspace.hr.trg_emp` is created with the correct Spark SQL data types and Delta Lake properties before running this notebook. The column names and order in the `INSERT` statement imply a specific target table structure.\n",
        "- **Initial Load vs. Incremental:** This script performs a full load. If `trg_emp` is meant to be incrementally loaded or merged, the logic would need to be enhanced with `MERGE INTO` or similar strategies based on business requirements. This conversion only reflects the `INSERT` provided in the source.\n",
        "- **Data Type Validation:** Verify that the data types of `workspace.hr.employees` columns match or are compatible with the target `workspace.hr.trg_emp` table. For example, Oracle `NUMBER(p,0)` typically maps to `BIGINT` in Spark, and `DATE` to `TIMESTAMP`."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "python",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}